# Troubleshooting Log — Phase 2 TAM Subtype Analysis

이 노트북은 Phase 2 파이프라인 구축 과정에서 발생한 문제들의 **발견 -> 원인 분석 -> 해결** 흐름을 기록한다.  
최종 파이프라인(`01_myeloid_subset.ipynb`, `02_DEG_analysis.ipynb`, `03_TME_composition.ipynb`)에 반영된 모든 설계 결정의 근거가 여기에 있다.

---

## Issues Index

| # | 문제 | 발견 시점 | 해결 방법 |
|---|------|----------|----------|
| 1 | Phase 1 전체 cell 기준 PCA embedding을 macrophage subset에 그대로 사용 | macrophage subset 재클러스터링 직후 | macrophage 전용 HVG/PCA/Harmony 재계산 |
| 2 | n_pcs=17을 macrophage subset에 검증 없이 사용 | subset 전용 PCA 재계산 시점 | subset 전용 variance ratio 분석 -> n_pcs=7 |
| 3 | resolution 선택 근거 없음 | annotation 검토 시점 | silhouette score sweep 수행 |
| 4 | 재클러스터링 후 cluster 번호 전면 교체 — 기존 annotation 맵 무효화 | 재클러스터링 결과 확인 시점 | 새 cluster 구조에서 annotation 재수행 |
| 5 | gene set score 절대값 비교의 통계적 한계 | cluster 5 annotation 불확실 발견 시점 | rank-based 비교로 전환 |
| 6 | adata.raw 오염 — HVG 이후 시점에 백업 | DEG 결과 이상 발견 시점 | use_raw=False + mac.X(log-normalized, full genes) 사용 |
| 7 | use_raw=True 설정 불일치 및 파라미터 미명시 | DEG 파라미터 검토 시점 | use_raw=False + bonferroni + tie_correct 명시 |
| 8 | DEG overlap 검증 시 logfoldchanges 기준 재정렬 오류 | overlap 결과 검토 시점 | rank_genes_groups_df 기본 ranking 유지 |
| 9 | Unknown -> Unresolved 라벨 의미 불명확 | annotation 결과 해석 시점 | 의미 분리 후 Unresolved로 명시적 재정의 |
| 10 | SPP1+(tentative) 귀속 불확실 | DEG overlap 비교 결과 시점 | DEG-B / DEG-C 추가 -> C1QC/SPP1 mixed-signature TAM으로 재분류 |
| 11 | TME composition에서 tam_subtype 컬럼 사용 — final re-annotation 미반영 | composition 재계산 시점 | tam_subtype_final 컬럼으로 전환 |
| 12 | TME composition 통합 로직 변경 — SPP1+(tentative) 취급 오류 | DEG-C 결과 반영 시점 | mixed-signature TAM 별도 유지 + C1QC계열만 통합 |
| 13 | TME composition에서 Unresolved 100% sample 원인 오판 | stacked bar 확인 시점 | 3단계 추적 -> framework 외부 state로 판단 |

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
from utils.paths import *
from utils.report import ph2_report_adata_state

---
## Issue 1 — Phase 1 전체 cell 기준 PCA embedding을 macrophage subset에 그대로 사용

### 현상
macrophage subset 재클러스터링 시 Phase 1에서 생성된 `X_pca_harmony`(전체 44,860개 세포 기준)를 그대로 사용하였다.  
`mac.uns['pca']['variance_ratio']`를 확인하니 전체 데이터 기준 분포와 동일하여 macrophage 내부 구조가 전체 cell variance에 묻혀 있음을 확인하였다.

### 원인
Phase 1에서 `adata.obsm['X_pca_harmony']`를 생성한 뒤 macrophage subset 시 `.copy()`로 해당 embedding이 그대로 상속되었다.  
전체 cell 기준 embedding에서는 T cell/B cell/tumor cell 간 차이가 주요 variance를 차지하기 때문에, macrophage 내부 subtype 간 차이가 이 variance에 묻힐 수 있다.

### 해결
macrophage subset 내부에서 filter_genes -> HVG selection(batch_key='sample') -> scale(PCA 입력용 임시 객체에만 적용) -> PCA -> Harmony를 순서대로 재계산하였다.  
`mac.X`는 log-normalized 상태로 유지하고 scale은 PCA 계산에만 사용하여, 이후 marker score/dotplot 등 expression 기반 분석에 영향이 없도록 하였다.

In [ ]:
# ❌ 문제 코드 — Phase 1 전체 cell 기준 PCA embedding 그대로 사용
mac = adata[adata.obs['cell_type'] == 'Macrophage'].copy()
# mac.obsm['X_pca_harmony']가 Phase 1 전체 cell 기준 좌표를 그대로 보유
# mac.uns['pca']['variance_ratio']도 전체 데이터 기준 -> macrophage 내부 구조 반영 안됨

sc.pp.neighbors(mac, use_rep='X_pca_harmony', n_pcs=17, random_state=42)
sc.tl.umap(mac, random_state=42)
sc.tl.leiden(mac, resolution=0.5, random_state=42)
# -> macrophage 내부 subtype 구조가 전체 cell variance에 묻혀 의미 없는 클러스터 생성 위험

In [ ]:
# ✅ 해결 코드 — macrophage subset 전용 HVG/PCA/Harmony 재계산
mac = adata[adata.obs['cell_type'] == 'Macrophage'].copy()

# 진단: Phase 1 raw가 정상 시점(log-normalized, scale 이전)인지 먼저 확인
assert adata.raw is not None
print(adata.raw.shape)         # (44860, 41861) - 전체 gene이어야 정상
print(adata.raw.X.min(), adata.raw.X.max())  # 0 이상이어야 정상

# 1. macrophage subset 내부에서 filter_genes (subset에서 발현 없는 유전자 제거)
sc.pp.filter_genes(mac, min_cells=3)

# 2. subset 전용 HVG selection (batch_key로 sample간 공통 HVG 선택)
sc.pp.highly_variable_genes(mac, n_top_genes=2000, batch_key='sample')
print('HVG count:', mac.var['highly_variable'].sum())
print('NaN in dispersions_norm:', mac.var['dispersions_norm'].isna().sum())

# 3. PCA는 HVG 기준 scaled 임시 객체에서만 계산 (mac.X는 log-normalized 상태 유지)
mac_pca_input = mac[:, mac.var['highly_variable']].copy()
sc.pp.scale(mac_pca_input, max_value=10)
sc.pp.pca(mac_pca_input, n_comps=50, random_state=42)
mac.obsm['X_pca'] = mac_pca_input.obsm['X_pca'].copy()
mac.uns['pca'] = mac_pca_input.uns['pca'].copy()
del mac_pca_input

# 4. macrophage 전용 Harmony 보정
import harmonypy as hm
pca_result = mac.obsm['X_pca'][:, :7]  # n_pcs는 Issue 2에서 확정
meta = mac.obs[['sample']]
ho = hm.run_harmony(pca_result, meta, 'sample')
Z_corr = ho.Z_corr
if Z_corr.shape[0] != mac.n_obs and Z_corr.shape[1] == mac.n_obs:
    Z_corr = Z_corr.T
assert Z_corr.shape[0] == mac.n_obs
mac.obsm['X_pca_harmony'] = Z_corr

# mac.X는 log-normalized 상태 유지 확인
print('mac.X min/max:', mac.X.min(), mac.X.max())

---
## Issue 2 — n_pcs=17을 macrophage subset에 검증 없이 사용

### 현상
Phase 1 전체 데이터 elbow 기준으로 설정한 `n_pcs=17`을 macrophage subset 재클러스터링에 그대로 사용하였다.  
macrophage subset 전용 PCA를 재계산하고 나니 variance ratio가 전체 데이터와 다른 분포를 보였다.

### 원인
전체 cell(44,860개)의 PCA elbow와 macrophage subset(~9,100개)의 PCA elbow는 다를 수 있다.  
세포 수와 transcriptional variance 구조 모두 다르므로 n_pcs를 재검증해야 한다.

### 해결
macrophage subset 전용 PCA에서 variance ratio를 계산하여 인접 PC 간 감소폭을 정량 확인하였다.  
PC1~PC7: 유의미한 변화, PC8 이후: 감소폭 -0.0001 수준에서 평평해짐 -> `n_pcs=7`로 확정하였다.  
또한 누적 variance가 낮은 이유(6~7% 수준)는 macrophage subset이 이미 균질한 population이기 때문으로 판단하였다.

In [ ]:
# ❌ 문제 코드 — Phase 1 n_pcs=17 그대로 사용
sc.pp.neighbors(mac, use_rep='X_pca_harmony', n_pcs=17, random_state=42)
# -> macrophage subset PCA에서 PC7 이후 이미 flat -> 불필요한 noise PC 포함

In [ ]:
# ✅ 해결 코드 — macrophage subset 전용 elbow 정량 분석
sc.pl.pca_variance_ratio(mac, n_pcs=50, log=True)
print(mac.uns['pca']['variance_ratio'][:20])

# 인접 PC 간 variance 감소폭 정량 확인
vr = mac.uns['pca']['variance_ratio']
diffs = np.diff(vr)
for i, d in enumerate(diffs[:25], start=1):
    print(f'PC{i} -> PC{i+1}: {d:.5f}')

# 누적 variance 확인
cumsum = np.cumsum(vr)
for i in [7, 10, 15, 20]:
    print(f'PC1~{i} 누적 variance: {cumsum[i-1]*100:.2f}%')

# 결과:
# PC1~PC7: 유의미한 감소 / PC8 이후: -0.0001 수준으로 평평
# 누적 variance ~6.56% — macrophage subset이 이미 균질한 population이라 정상 범주
# -> n_pcs=7로 확정
sc.pp.neighbors(mac, use_rep='X_pca_harmony', n_pcs=7, random_state=42)

---
## Issue 3 — resolution 선택 근거 없음

### 현상
Harmony를 적용하기 전 임의로 `resolution=0.5`를 사용하였고, Harmony 적용 후에도 특별한 정량적 근거 없이 동일 값을 유지하였다.

### 원인
resolution은 leiden clustering의 세분화 정도를 결정하는 핵심 하이퍼파라미터인데, '논문에서 흔히 쓰이는 값' 수준으로 설정하면 downstream annotation의 신뢰도가 낮아진다.

### 해결
silhouette score sweep(resolution 0.1~1.0)을 수행하여 클러스터 분리도를 정량 확인하였다.  
최고점은 resolution=0.3~0.4(silhouette 0.226)였으나, 해당 resolution에서는 C1QC core와 SPP1 core cluster가 하나로 뭉쳐 subtype 구분이 불가능하였다.  
TAM subtype annotation 목적상 core cluster 분리가 명확한 `resolution=0.5`를 채택하고, silhouette 최고점과의 trade-off 이유를 명시하였다.  
또한 Phase 1 cell type annotation의 resolution 민감도도 사후 검증하여 Phase 2 재클러스터링이 Phase 1 결과에 영향을 주지 않음을 확인하였다.

In [ ]:
# ❌ 문제 코드 — 근거 없이 resolution=0.5 사용
sc.tl.leiden(mac, resolution=0.5, random_state=42)
# -> TAM subtype 분리 가능한 resolution인지 검증하지 않음

In [ ]:
# ✅ 해결 코드 — silhouette score sweep
from sklearn.metrics import silhouette_score

resolutions = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
results = []

for res in resolutions:
    sc.tl.leiden(mac, resolution=res, random_state=42, key_added=f'leiden_r{res}')
    labels = mac.obs[f'leiden_r{res}'].astype(int).values
    emb = mac.obsm['X_pca_harmony']
    if len(set(labels)) > 1:
        score = silhouette_score(emb, labels, sample_size=3000, random_state=42)
    else:
        score = float('nan')
    results.append({'resolution': res, 'n_clusters': len(set(labels)), 'silhouette': round(score, 4)})

display(pd.DataFrame(results))

# 결과:
# 0.3~0.4: silhouette 최고점 0.226 — 그러나 C1QC/SPP1 core가 하나의 cluster로 뭉침
# 0.5: silhouette 0.200으로 낮아지지만 C1QC/SPP1 core cluster 분리 확인
# -> 수치 최고점(0.3~0.4)과 annotation 목적(0.5) 간 trade-off를 명시하고 resolution=0.5 채택

In [ ]:
# ✅ Phase 1 cell type annotation resolution 민감도 사후 검증
# Phase 2에서 macrophage subset resolution을 재검토하면서,
# Phase 1 cell type annotation도 resolution 선택에 영향받는지 우려가 있어 추가 검증

adata = sc.read_h5ad(HUMAN_H5AD)
sc.pp.neighbors(adata, use_rep='X_pca_harmony', n_pcs=17, random_state=42)

for res in [0.3, 0.4, 0.5, 0.6, 0.7]:
    sc.tl.leiden(adata, resolution=res, random_state=42, key_added=f'leiden_ph1_r{res}')
    print(f'resolution={res}: {adata.obs[f"leiden_ph1_r{res}"].nunique()} clusters')

# 결과: Phase 1은 X_pca_harmony(n_pcs=17) 기준으로 resolution 0.3~0.7 모두
#       동일한 cell type 경계를 유지 -> Phase 2 재클러스터링이 Phase 1 결과에 영향 없음 확인

---
## Issue 4 — 재클러스터링 후 cluster 번호 전면 교체 — 기존 annotation 맵 무효화

### 현상
OLD에서 설정한 cluster->subtype 매핑:
```python
{'6': 'C1QC+ TAM', '0': 'C1QC+ TAM', '1': 'C1QC+ TAM(tentative)',
 '9': 'SPP1+ TAM', '5': 'SPP1+ TAM(tentative)'}
```
Issue 1(embedding 재계산)을 해결하고 재클러스터링하니 cluster 번호가 전면 교체되었다.
```python
{'0': 'C1QC+ TAM', '2': 'C1QC+ TAM(tentative)',
 '4': 'SPP1+ TAM', '5': 'SPP1+ TAM(tentative)'}
```

### 원인
embedding이 전면 재계산되면 neighbor graph 구조가 달라지고, leiden cluster 번호는 graph 구조 및 random_state에 의해 결정되므로 이전 번호가 그대로 유지되지 않는다.

### 해결
cluster 번호를 하드코딩으로 유지하지 않고, 재클러스터링 후 UMAP + dotplot + gene set score를 새로 확인하여 annotation을 처음부터 재수행하였다.  
cluster 번호 자체보다 각 cluster의 marker expression 패턴으로 annotation하는 것이 재현 가능한 방식이다.

In [ ]:
# ❌ 문제 코드 — 이전 cluster 번호 하드코딩 그대로 사용
mac.obs['tam_subtype'] = mac.obs['leiden'].map({
    '6': 'C1QC+ TAM',       # <- 재클러스터링 후 cluster 6이 더이상 C1QC core가 아닐 수 있음
    '0': 'C1QC+ TAM',
    '1': 'C1QC+ TAM(tentative)',
    '9': 'SPP1+ TAM',
    '5': 'SPP1+ TAM(tentative)',
}).fillna('Unknown')

In [ ]:
# ✅ 해결 코드 — 재클러스터링 후 annotation 재수행

# 1. 새 cluster 구조 확인
print('Leiden cluster count:', mac.obs['leiden'].nunique())
print(mac.obs['leiden'].value_counts().sort_index())

# 2. UMAP + marker 발현 확인 (cluster 번호 없이 expression pattern 기준으로 재확인)
sc.pl.umap(mac, color=['leiden', 'C1QC', 'SPP1', 'ISG15'], ncols=2)

# 3. gene set score 계산 후 rank 기반 annotation (Issue 5 참조)
# 확인 결과 새 cluster 번호:
# cluster 0 -> C1QC+ TAM core
# cluster 2 -> C1QC+ TAM(tentative)
# cluster 4 -> SPP1+ TAM core
# cluster 5 -> SPP1+ TAM(tentative) [-> DEG-C에서 mixed-signature로 재분류]
# 나머지   -> Unresolved

mac.obs['tam_subtype'] = mac.obs['leiden'].map({
    '0': 'C1QC+ TAM',
    '2': 'C1QC+ TAM(tentative)',
    '4': 'SPP1+ TAM',
    '5': 'SPP1+ TAM(tentative)',
}).fillna('Unresolved')  # Unknown -> Unresolved (Issue 9 참조)

---
## Issue 5 — gene set score 절대값 비교의 통계적 한계

### 현상
초기 annotation에서 cluster별 `C1QC_score`와 `SPP1_score` 절대값을 직접 비교하여 서브타입을 결정하였다.  
cluster 5의 경우 `C1QC_score`의 절대값이 `SPP1_score`보다 높음에도 불구하고 SPP1 axis가 상대적으로 높다는 이유로 SPP1+(tentative)로 분류하는 판단 오류가 발생하였다.

### 원인
C1QC_score와 SPP1_score는 서로 다른 marker set으로 계산되어 scale이 다를 수 있다.  
`score_genes`는 marker gene 평균에서 비슷한 발현 수준의 control gene 평균을 뺀 값이므로,  
두 score의 절대값 크기를 직접 비교하는 것은 scale이 다를 경우 왜곡된 판단을 유발한다.

### 해결
논문 Table S3 기준 4개 marker set(Resting_C1QC / Activated_C1QC / SPP1 / ISG15)을 모두 계산하고,  
각 marker set 내에서 cluster 간 rank를 계산하여 비교하였다.  
즉 '이 cluster가 C1QC_score에서 몇 위인지', 'SPP1_score에서 몇 위인지'를 각각 비교함으로써 scale 차이 영향을 제거하였다.

In [ ]:
# ❌ 문제 코드 — 절대값 직접 비교 + 단순 C1QC/SPP1 두 개만 비교
sc.tl.score_genes(mac, ['C1QC', 'APOE', 'APOC1', 'SELENOP'], score_name='C1QC_score')
sc.tl.score_genes(mac, ['SPP1', 'GPNMB', 'CTSD', 'MRC1', 'CD63'], score_name='SPP1_score')

summary = mac.obs.groupby('leiden')[['C1QC_score', 'SPP1_score']].mean()
# -> 두 score의 scale이 다를 수 있어 절대값 직접 비교 부적절
# -> cluster 5: C1QC_score=0.12 > SPP1_score=0.09 임에도 SPP1 방향으로 오판 가능

In [ ]:
# ✅ 해결 코드 — 논문 Table S3 기준 4개 marker set + rank-based 비교

# 논문 Table S3 상위 marker 기준으로 4개 score 계산
resting_c1qc_markers  = ['SELENOP', 'APOC1', 'C1QB', 'C1QA', 'C1QC']
activated_c1qc_markers = ['SELENOP', 'FOLR2', 'SLC40A1', 'RNASE1', 'PLTP', 'DAB2', 'C1QC']
spp1_markers           = ['FN1', 'SPP1', 'INHBA', 'CXCL3', 'FABP5', 'MARCO']
isg15_markers          = ['CXCL10', 'CXCL11', 'CXCL9', 'IDO1', 'IFIT1', 'RSAD2']

for name, genes in [
    ('Resting_C1QC_score', resting_c1qc_markers),
    ('Activated_C1QC_score', activated_c1qc_markers),
    ('SPP1_score', spp1_markers),
    ('ISG15_score', isg15_markers),
]:
    valid = [g for g in genes if g in mac.var_names]
    if valid:
        sc.tl.score_genes(mac, valid, score_name=name)

score_cols = ['Resting_C1QC_score', 'Activated_C1QC_score', 'SPP1_score', 'ISG15_score']
summary = mac.obs.groupby('leiden')[score_cols].mean()

# 각 score 내에서 cluster 간 rank 계산 (1 = 해당 score에서 가장 높은 cluster)
rank_df = summary.rank(ascending=False).astype(int)
print('=== Absolute score ===')
print(summary.round(3))
print('\n=== Rank within each score ===')
print(rank_df)

---
## Issue 6 — adata.raw 오염 — HVG 이후 시점에 백업

### 현상
DEG 분석에서 `use_raw=True`로 실행했는데, SPP1+ TAM의 논문 signature overlap이 1/9(11.1%)로 나왔다.  
특히 SPP1 자체가 DEG ranking 49위로 밀려 있었다. 논문 dotplot에서 SPP1이 해당 population에서 명확히 상위 발현이어야 하므로 이상함을 인지하였다.

### 원인
`mac.raw`가 설정된 시점을 확인하니 Phase 1에서 `adata.raw = adata`를 HVG selection 이후에 실행하였다.  
이에 따라 adata.raw(그리고 이를 상속한 mac.raw)가 HVG 2,000개만 포함한 상태였다.  
DEG에서 `use_raw=True`를 사용하면 전체 32,634개 gene이 아니라 2,000개 HVG 기준으로 ranking하기 때문에  
SPP1 등 HVG에 포함되지 않거나 우선순위가 낮은 유전자가 DEG 상위권에서 밀려난다.

### 해결
mac.X가 Issue 1의 재구축 과정에서 log-normalization만 적용된 full gene(32,634개) matrix임을 확인하였다.  
`use_raw=False`로 변경하여 mac.X 기준으로 DEG를 재실행하였다.  
수정 후 SPP1+ TAM overlap이 7/9(77.8%)로 회복되고, SPP1이 2위 DEG로 확인되었다.

> Phase 1 troubleshooting log Issue 7(HVG 필터 후 raw backup 위치 오류)과 연결된 downstream 영향이다.

In [ ]:
# 오염 진단 코드
print('mac.raw exists:', mac.raw is not None)
if mac.raw is not None:
    print('mac.raw.shape:', mac.raw.shape)   # (9106, 2000) <- HVG만 포함 -> 오염
print('mac.X shape:', mac.shape)             # (9106, 32634) 가 정상
print('mac.X min/max:', mac.X.min(), mac.X.max())  # log-normalized이면 0~10 수준

In [ ]:
# ❌ 문제 코드 — use_raw=True + 오염된 raw
sc.tl.rank_genes_groups(
    mac,
    groupby='tam_subtype',
    method='wilcoxon',
    key_added='deg_tam',
    use_raw=True  # mac.raw가 HVG 2,000개만 포함한 상태
)
# -> SPP1+ TAM overlap 1/9(11.1%), SPP1 자체가 49위로 밀림

In [ ]:
# ✅ 해결 코드 — use_raw=False + mac.X(log-normalized, 32,634 genes)
print('mac shape:', mac.shape)               # (9106, 32634) 확인
print('mac.X min/max:', mac.X.min(), mac.X.max())

sc.tl.rank_genes_groups(
    mac,
    groupby='tam_subtype',
    groups='all',
    reference='rest',
    method='wilcoxon',
    corr_method='bonferroni',
    tie_correct=True,
    use_raw=False,   # mac.X 기준 (log-normalized, full genes)
    pts=True,
    key_added='deg_subtype_vs_rest',
)
# -> SPP1+ TAM overlap 7/9(77.8%), SPP1 2위로 회복

---
## Issue 7 — use_raw=True 설정 불일치 및 DEG 파라미터 미명시

### 현상
초기 DEG 코드에서 `corr_method`, `tie_correct`, `pts` 파라미터가 명시되지 않았다.  
Scanpy 기본값에 의존하여 multiple testing correction 방식과 동점 처리 방식이 코드만 보고는 확인 불가능한 상태였다.

### 원인
DEG 실행 파라미터를 명시적으로 선언하지 않으면 Scanpy 버전 업데이트 시 기본값이 바뀔 수 있다.  
scRNA-seq에서 cell 수 편차가 큰 경우 tie_correct 없이는 Wilcoxon rank-sum 결과가 불안정해질 수 있다.

### 해결
모든 DEG 실행(DEG-A, DEG-B, DEG-C)에서 파라미터를 명시적으로 선언하였다.

In [ ]:
# ❌ 문제 코드 — 파라미터 미명시
sc.tl.rank_genes_groups(
    mac,
    groupby='tam_subtype',
    method='wilcoxon',
    key_added='deg_tam',
    use_raw=True
    # corr_method: 기본값 의존 (버전마다 다를 수 있음)
    # tie_correct: 기본값 의존
    # pts: False (pct.1/pct.2 출력 안됨)
)

In [ ]:
# ✅ 해결 코드 — 모든 파라미터 명시적 선언 (DEG-A / DEG-B / DEG-C 공통 기준)

# DEG-A: subtype vs rest
sc.tl.rank_genes_groups(
    mac,
    groupby='tam_subtype',
    groups='all',
    reference='rest',
    method='wilcoxon',
    corr_method='bonferroni',  # multiple testing correction 명시
    tie_correct=True,           # 동점 처리 명시
    use_raw=False,              # mac.X 기준
    pts=True,                   # pct.1/pct.2 출력
    key_added='deg_subtype_vs_rest',
)

# DEG-B: core 간 직접 비교 시에도 동일 파라미터 기준 사용
# DEG-C: ambiguous 비교 시에도 동일 파라미터 기준 사용

---
## Issue 8 — DEG overlap 검증 시 logfoldchanges 기준 재정렬 오류

### 현상
초기 DEG overlap 검증 코드에서 `rank_genes_groups_df` 결과를 `logfoldchanges` 기준으로 재정렬한 뒤 top 50 gene을 추출하였다.  
이 경우 Scanpy의 Wilcoxon score 기반 ranking 순서가 변경되어 원래 DEG 상위권 유전자가 밀리는 문제가 발생하였다.

### 원인
Scanpy의 `rank_genes_groups_df`는 이미 Wilcoxon test score(z-statistic) 기준으로 ranking된 결과를 반환한다.  
`logfoldchanges`는 fold change 크기이고, score(통계적 유의성)와 반드시 같은 순서가 아니다.  
fold change로 재정렬하면 p-value가 낮아도 fold change가 큰 유전자가 상위로 올라올 수 있어 DEG ranking 해석이 달라진다.

### 해결
`rank_genes_groups_df` 결과에 추가 정렬을 수행하지 않고 Scanpy 기본 ranking을 유지하여 top 50 gene을 추출하도록 수정하였다.

In [ ]:
# ❌ 문제 코드 — logfoldchanges 기준 재정렬 후 top N 추출
for group in mac.obs['tam_subtype'].cat.categories:
    deg = sc.get.rank_genes_groups_df(mac, group=group, key='deg_subtype_vs_rest').copy()
    deg = deg.sort_values('logfoldchanges', ascending=False)  # <- Wilcoxon ranking 순서 변경
    my_top = deg.head(50)
    # -> fold change 큰 유전자가 올라오고, p-value 기준 상위 유전자가 밀릴 수 있음

In [ ]:
# ✅ 해결 코드 — rank_genes_groups_df 기본 ranking 유지
for group in mac.obs['tam_subtype'].cat.categories:
    deg = sc.get.rank_genes_groups_df(mac, group=group, key='deg_subtype_vs_rest').copy()
    # rank_genes_groups_df는 이미 Wilcoxon score 기준으로 정렬되어 있음
    # 추가 sort 없이 head(50)으로 상위 DEG 추출
    deg['gene_clean'] = deg['names'].map(lambda x: str(x).strip().upper())
    deg['rank'] = deg.index + 1
    my_top = deg.head(50).copy()
    # -> Wilcoxon score 기반 ranking 그대로 유지

---
## Issue 9 — Unknown -> Unresolved 라벨 의미 불명확

### 현상
초기 annotation에서 C1QC/SPP1 어느 subtype에도 해당하지 않는 cluster를 `Unknown`으로 라벨링하였다.

### 원인
`Unknown`은 '분류를 시도했으나 결론을 내지 못한 상태'를 의미할 수도 있고, '분석하지 않은 상태'로 해석될 수도 있어 의미가 불명확하다.  
실제로는 C1QC/SPP1 marker가 낮고 FCN1, CFP, SELL, CORO1A 등 monocyte-like marker가 높은 distinct population이어서 '알 수 없음'이 아니라 '현재 framework으로 분류 불가'가 더 정확한 표현이다.

### 해결
해당 cluster의 이름을 `Unresolved`로 변경하였다. Unresolved는 "C1QC/SPP1 subtype framework 내에서 어느 쪽으로도 안정적으로 귀속되지 않는 myeloid population"임을 명시하는 라벨이다.  
이 population의 top DEG(FCN1, CFP, SELL, CORO1A)는 monocyte-like / inflammatory myeloid feature와 더 가까웠다.

In [ ]:
# ❌ 문제 코드 — Unknown 라벨 사용
mac.obs['tam_subtype'] = mac.obs['leiden'].map({
    '6': 'C1QC+ TAM',
    '0': 'C1QC+ TAM',
    '1': 'C1QC+ TAM(tentative)',
    '9': 'SPP1+ TAM',
    '5': 'SPP1+ TAM(tentative)',
}).fillna('Unknown')  # <- 의미 불명확

In [ ]:
# ✅ 해결 코드 — Unresolved 라벨로 변경 + 의미 명시
# Unresolved: 현재 C1QC/SPP1 framework으로 안정적으로 분류되지 않는 myeloid population
# (FCN1, CFP, SELL, CORO1A 등 monocyte-like feature 우세)
mac.obs['tam_subtype'] = mac.obs['leiden'].map({
    '0': 'C1QC+ TAM',
    '2': 'C1QC+ TAM(tentative)',
    '4': 'SPP1+ TAM',
    '5': 'SPP1+ TAM(tentative)',
}).fillna('Unresolved')  # <- C1QC/SPP1 framework 외부 population임을 명시

print(mac.obs['tam_subtype'].value_counts())

---
## Issue 10 — SPP1+(tentative) 귀속 불확실

### 현상
`SPP1+ TAM(tentative)`로 분류된 population이 DEG-A(vs rest) 기준에서는 marker score상 SPP1 axis가 근소하게 높게 나타났으나,  
reference signature overlap 결과에서는 SPP1+ TAMs 2/9(22.2%)보다 Resting C1QC+ TAMs 7/24(29.2%)가 더 높게 나왔다.  
annotation 근거인 marker score와 DEG overlap 결과가 상충하여 최종 subtype 귀속이 불확실하였다.

### 원인
DEG-A(vs rest)는 해당 population을 전체 macrophage population 대비 비교하므로, SPP1과 C1QC 특성이 혼재하는 population에서는 어느 쪽 특성이 dominant한지 판단하기 어렵다.  
두 core population 중 어느 쪽에 더 가까운지 판단하려면 직접 비교가 필요하다.

### 해결
**DEG-B**: C1QC+ TAM core vs SPP1+ TAM core 직접 비교 -> 두 core의 transcriptional 차이가 direct comparison에서도 유지됨을 확인  
**DEG-C**: ambiguous vs SPP1 core / ambiguous vs C1QC tentative 양방향 비교
- ambiguous > SPP1 core: C1QC-associated feature 우세 (C1QB, C1QA, APOE, TREM2, CTSD, GPNMB)
- ambiguous > C1QC tentative: SPP1-associated feature 우세 (SPP1, FABP5, MARCO, SDC2, FN1, INHBA)

-> 어느 한 subtype으로 단순 분류하지 않고 `C1QC/SPP1 mixed-signature TAM`으로 최종 재어노테이션  
-> 초기 annotation 백업(`tam_subtype_initial`) 보존

In [ ]:
# ❌ 문제 코드 — DEG-A 결과 + marker score만으로 SPP1+(tentative) 유지
# SPP1+ TAM(tentative) overlap:
# SPP1+ TAMs:       2/9  (22.2%)
# Resting C1QC+ TAMs: 7/24 (29.2%)  <- 실은 C1QC 쪽이 더 높음
# marker score에서 SPP1 axis가 근소하게 높다는 이유만으로 SPP1+(tentative) 유지
# -> 근거 불충분한 상태로 annotation 확정

In [ ]:
# ✅ 해결 코드 — DEG-B: C1QC core vs SPP1 core 직접 비교
CORE_C1QC = 'C1QC+ TAM'
CORE_SPP1 = 'SPP1+ TAM'

core_tam = mac[mac.obs['tam_subtype'].isin([CORE_C1QC, CORE_SPP1])].copy()
print(core_tam.obs['tam_subtype'].value_counts())

sc.tl.rank_genes_groups(
    core_tam, groupby='tam_subtype',
    groups=[CORE_C1QC], reference=CORE_SPP1,
    method='wilcoxon', corr_method='bonferroni',
    tie_correct=True, use_raw=False, pts=True,
    key_added='deg_c1qc_vs_spp1_core'
)
sc.tl.rank_genes_groups(
    core_tam, groupby='tam_subtype',
    groups=[CORE_SPP1], reference=CORE_C1QC,
    method='wilcoxon', corr_method='bonferroni',
    tie_correct=True, use_raw=False, pts=True,
    key_added='deg_spp1_vs_c1qc_core'
)
# 결과:
# C1QC > SPP1: SELENOP, C1QA/B/C, PLTP, FOLR2, SLC40A1, F13A1, LGMN
# SPP1 > C1QC: SPP1, MCEMP1, RETN, VCAN, FCN1, CLEC5A, S100A8

In [ ]:
# ✅ 해결 코드 — DEG-C: ambiguous population 양방향 비교
AMBIGUOUS     = 'SPP1+ TAM(tentative)'
C1QC_TENT     = 'C1QC+ TAM(tentative)'
SPP1_CORE     = 'SPP1+ TAM'

amb_tam = mac[mac.obs['tam_subtype'].isin([AMBIGUOUS, C1QC_TENT, SPP1_CORE])].copy()

# 비교 1: ambiguous vs SPP1 core
sc.tl.rank_genes_groups(
    amb_tam, groupby='tam_subtype',
    groups=[AMBIGUOUS], reference=SPP1_CORE,
    method='wilcoxon', corr_method='bonferroni',
    tie_correct=True, use_raw=False, pts=True,
    key_added='deg_ambiguous_vs_spp1_core'
)
# -> ambiguous에서 C1QB, C1QA, APOE, TREM2, CTSD, GPNMB 높음 (C1QC feature)

# 비교 2: ambiguous vs C1QC tentative
sc.tl.rank_genes_groups(
    amb_tam, groupby='tam_subtype',
    groups=[AMBIGUOUS], reference=C1QC_TENT,
    method='wilcoxon', corr_method='bonferroni',
    tie_correct=True, use_raw=False, pts=True,
    key_added='deg_ambiguous_vs_c1qc_tentative'
)
# -> ambiguous에서 SPP1, FABP5, MARCO, SDC2, FN1, INHBA 높음 (SPP1 feature)

# 결론: 어느 쪽도 아님 -> C1QC/SPP1 mixed-signature TAM으로 재분류
mac.obs['tam_subtype_initial'] = mac.obs['tam_subtype'].copy()  # 초기 annotation 백업
mac.obs['tam_subtype_final'] = (
    mac.obs['tam_subtype']
    .astype(str)
    .replace({'SPP1+ TAM(tentative)': 'C1QC/SPP1 mixed-signature TAM'})
)
print(mac.obs['tam_subtype_final'].value_counts())

---
## Issue 11 — TME composition에서 tam_subtype 컬럼 사용 — final re-annotation 미반영

### 현상
OLD의 TME composition 코드에서 `required_cols = ['sample', 'tam_subtype']`을 사용하였다.  
그러나 DEG-C 이후 `SPP1+ TAM(tentative)` -> `C1QC/SPP1 mixed-signature TAM`으로 재분류된 결과가 `tam_subtype_final` 컬럼에 저장되어 있었으므로, OLD 코드를 그대로 사용하면 final re-annotation 결과가 반영되지 않는다.

### 원인
DEG-A 단계에서 사용하던 `tam_subtype` 컬럼이 DEG-C 이후 `tam_subtype_final`로 업데이트되었는데, downstream 노트북(03_TME_composition.ipynb)이 이를 자동으로 따라가지 않아 컬럼 불일치가 발생하였다.

### 해결
`required_cols = ['sample', 'tam_subtype_final']`로 변경하여 DEG-C 기반 final re-annotation 결과를 TME composition에 반영하였다.

In [ ]:
# ❌ 문제 코드 — tam_subtype 컬럼 사용 (DEG-C 재분류 결과 미반영)
required_cols = ['sample', 'tam_subtype']
missing_cols = [c for c in required_cols if c not in mac.obs.columns]
print('missing required columns:', missing_cols)

composition = (
    mac.obs
    .groupby(['sample', 'tam_subtype'], observed=False)
    .size()
    .unstack(fill_value=0)
)
# -> SPP1+ TAM(tentative)가 DEG-C에서 mixed-signature로 재분류됐는데
#    tam_subtype 컬럼에는 여전히 SPP1+ TAM(tentative)로 남아있음

In [ ]:
# ✅ 해결 코드 — tam_subtype_final 컬럼으로 전환
required_cols = ['sample', 'tam_subtype_final']  # DEG-C 재분류 결과 반영
missing_cols = [c for c in required_cols if c not in mac.obs.columns]
print('missing required columns:', missing_cols)

composition = (
    mac.obs
    .groupby(['sample', 'tam_subtype_final'], observed=False)
    .size()
    .unstack(fill_value=0)
)
composition_pct = composition.div(composition.sum(axis=1), axis=0) * 100
print(composition_pct.round(1))

---
## Issue 12 — TME composition 통합 로직 변경 — SPP1+(tentative) 취급 오류

### 현상
OLD의 TME composition에서 tentative subtype 통합 로직:
```python
mac.obs['tam_subtype_final'] = mac.obs['tam_subtype'].replace({
    'C1QC+ TAM(tentative)': 'C1QC+ TAM',
    'SPP1+ TAM(tentative)': 'SPP1+ TAM',   # <- DEG-C 결과를 무시하고 단순 병합
})
```
이 코드는 DEG-C에서 `SPP1+ TAM(tentative)`가 mixed-signature임을 밝혔음에도 불구하고, SPP1+ TAM으로 단순 병합하는 오류를 만든다.

### 원인
DEG-C 이전에 작성된 통합 로직이 DEG-C 결과를 반영하지 않고 그대로 유지되었다.

### 해결
DEG-C 결과를 반영하여 통합 로직을 수정하였다.
- `C1QC+ TAM` + `C1QC+ TAM(tentative)` -> `C1QC-associated TAM` 으로 통합 (DEG-B에서 둘 다 C1QC program 확인)
- `C1QC/SPP1 mixed-signature TAM`은 별도 유지 (DEG-C에서 어느 쪽도 아님 확인)
- `SPP1+ TAM core`는 독립 유지
- `Unresolved`는 독립 유지

In [ ]:
# ❌ 문제 코드 — DEG-C 결과 무시, SPP1+(tentative)를 SPP1+ TAM으로 단순 병합
mac.obs['tam_subtype_final'] = (
    mac.obs['tam_subtype']
    .replace({
        'C1QC+ TAM(tentative)': 'C1QC+ TAM',
        'SPP1+ TAM(tentative)': 'SPP1+ TAM',  # <- DEG-C에서 mixed-signature임이 밝혀졌는데 무시
    })
)
# -> C1QC/SPP1 혼재 population을 SPP1+ TAM으로 잘못 분류

In [ ]:
# ✅ 해결 코드 — DEG-C 결과 반영한 통합 로직
# tam_subtype_final은 이미 DEG-C에서 생성됨:
# SPP1+ TAM(tentative) -> C1QC/SPP1 mixed-signature TAM

# TME composition용 merged label:
# C1QC+ TAM core + C1QC+ TAM(tentative) -> C1QC-associated TAM (DEG-B에서 둘 다 C1QC program 확인)
# SPP1+ TAM core: 독립 유지
# C1QC/SPP1 mixed-signature TAM: 독립 유지 (어느 쪽도 아님)
# Unresolved: 독립 유지

mac.obs['tam_subtype_merged'] = (
    mac.obs['tam_subtype_final']
    .replace({
        'C1QC+ TAM': 'C1QC-associated TAM',
        'C1QC+ TAM(tentative)': 'C1QC-associated TAM',
        # 'SPP1+ TAM': 독립 유지
        # 'C1QC/SPP1 mixed-signature TAM': 독립 유지
        # 'Unresolved': 독립 유지
    })
)

print(mac.obs['tam_subtype_merged'].value_counts())

---
## Issue 13 — TME composition에서 Unresolved 100% sample 원인 오판

### 현상
stacked bar plot에서 sample 4, 5, 6, 9, 13, 17, 22, 25가 Unresolved 100%로 나타났다.  
초기에는 'annotation이 너무 과하게 배제하는 것 아닌가' -> tentative를 통합하면 해결될 것이라고 가정하였다.  
그러나 tentative 통합 후에도 동일하게 Unresolved 100%가 유지되었다.

### 원인 분석 과정
1. 해당 sample들의 macrophage 세포 수를 확인 -> 38~331개로 세포 부족 문제가 아님
2. 해당 sample들의 leiden cluster 분포 확인 -> C1QC/SPP1 core cluster에 해당하는 cluster가 아예 없거나 극소수
3. 전체 adata에서 sample별 cell type 비율 확인 -> 해당 sample들은 macrophage 비율이 낮거나 T cell dominant

### 결론
Unresolved 100%는 annotation 오류가 아니라, 해당 sample의 macrophage들이 현재 C1QC/SPP1 framework으로 포착되지 않는 transcriptional state를 가진다는 뜻이다.  
tentative 통합으로 해결되지 않는 이유도 이 때문이며, Unresolved 유지가 더 정확한 해석이다.

In [ ]:
# ❌ 초기 접근 — annotation 문제로 오판하고 tentative 통합 시도
# composition plot에서 Unresolved가 많음 -> annotation 배제가 과하다?
# tentative 통합하면 해결될 것이라고 가정
# -> 통합 후에도 동일하게 Unresolved 100% 유지 -> 방향이 잘못됨

In [ ]:
# ✅ 해결 코드 — 3단계 원인 추적

# Step 1: 세포 수 확인 (세포 부족 문제인지?)
cell_counts = mac.obs.groupby('sample', observed=True).size()
print('Macrophage 세포 수 (전체 sample):')
print(cell_counts.sort_values())
# -> 38~331개 존재 -> 세포 수 부족 문제 아님

# Step 2: Unresolved 100% sample의 leiden cluster 분포 확인
unresolved_samples = ['4', '5', '6', '9', '13', '17', '22', '25']
mask = mac.obs['sample'].isin(unresolved_samples)
print('\nUnresolved 100% sample의 leiden 분포:')
print(mac.obs.loc[mask, 'leiden'].value_counts())
# -> C1QC core(cluster 0) / SPP1 core(cluster 4)가 없거나 극소수

# Step 3: 전체 adata에서 sample별 cell type 비율 확인 (macrophage 비율 자체가 낮은지?)
adata = sc.read_h5ad(HUMAN_H5AD)
cell_comp = (
    adata.obs
    .groupby(['sample', 'cell_type'], observed=False)
    .size()
    .unstack(fill_value=0)
)
cell_comp_pct = cell_comp.div(cell_comp.sum(axis=1), axis=0) * 100
display(cell_comp_pct.round(1))
# -> 해당 sample들은 T cell dominant 또는 B cell dominant
#    macrophage가 있더라도 C1QC/SPP1 framework 외부 state를 보이는 population

# 결론: annotation 문제 아님 -> Unresolved 유지가 올바른 해석

---
## Summary: 최종 파이프라인에 반영된 설계 결정

| 설계 결정 | 근거 (이슈 번호) |
|----------|----------------|
| macrophage subset 전용 HVG/PCA/Harmony 재계산 | Issue 1: Phase 1 embedding 상속 문제 |
| n_pcs=7 (macrophage subset 기준) | Issue 2: subset 전용 PCA elbow 분석 |
| resolution=0.5 (silhouette 검증 + core 분리 trade-off 명시) | Issue 3: silhouette score sweep |
| 재클러스터링 후 annotation 처음부터 재수행 | Issue 4: cluster 번호 전면 교체 |
| rank-based gene set score 비교 + 논문 4개 marker set | Issue 5: 절대값 비교의 scale 문제 |
| use_raw=False + mac.X(32,634 genes) 사용 | Issue 6: adata.raw 오염(HVG 이후 백업) |
| corr_method='bonferroni', tie_correct=True, pts=True 명시 | Issue 7: DEG 파라미터 불명확 |
| rank_genes_groups_df 기본 ranking 유지 (logfoldchanges 재정렬 금지) | Issue 8: DEG overlap ranking 변조 |
| Unknown → Unresolved (C1QC/SPP1 framework 외부 population 명시) | Issue 9: 라벨 의미 불명확 |
| DEG-B/C 추가 + C1QC/SPP1 mixed-signature TAM + tam_subtype_initial 백업 | Issue 10: ambiguous population 귀속 불확실 |
| tam_subtype_final 컬럼으로 전환 | Issue 11: final re-annotation 미반영 |
| SPP1+(tentative)를 SPP1+ TAM으로 단순 병합하지 않고 mixed-signature 별도 유지 | Issue 12: 통합 로직 오류 |
| Unresolved 100% sample = framework 외부 state (annotation 오류 아님) | Issue 13: 원인 오판 방지 |